# Task 9 — Industrial Predictive Maintenance

Train a binary machine-failure classifier on the UCI AI4I 2020 dataset. The notebook downloads data directly from UCI, tunes the decision threshold on validation data to control false discoveries, evaluates once on a held-out test set, and saves the complete inference pipeline.

**Important:** AI4I provides a binary `Machine failure` target and separate failure-mode indicators; it is not a mutually exclusive five-class target. This notebook models the binary target.

In [ ]:
# Install project dependencies
!pip -q install -r https://raw.githubusercontent.com/Desire-in-tech/predictive-maintenance-ml/task-09/predictive-maintenance/requirements.txt


In [ ]:
# Clone the project branch so the notebook always uses the committed source files
!git clone -q -b task-09/predictive-maintenance https://github.com/Desire-in-tech/predictive-maintenance-ml.git /content/predictive-maintenance-ml
%cd /content/predictive-maintenance-ml
import sys, os
sys.path.insert(0, '/content/predictive-maintenance-ml')


In [ ]:
# Download the dataset and inspect the target balance
from src.data_loader import load_ai4i, FEATURES, TARGET
import pandas as pd
df = load_ai4i()
display(df.head())
display(df[TARGET].value_counts().rename('count').to_frame())
print(f'Rows: {len(df):,}; failure prevalence: {df[TARGET].mean():.3%}')


In [ ]:
# Exploratory analysis: target distribution and sensor correlations
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(5, 3)); df[TARGET].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set(title='Machine failure class balance', xlabel='Machine failure (0=no, 1=yes)', ylabel='Rows'); plt.show()
plt.figure(figsize=(8, 6)); sns.heatmap(df.select_dtypes('number').corr(), cmap='coolwarm', center=0); plt.title('Numeric feature correlations'); plt.show()


In [ ]:
# Train, select a validation threshold, evaluate on held-out test data, and export artifact
!python -m src.train --output-dir artifacts --model random_forest --min-recall 0.50


In [ ]:
# Review the test metrics and saved feature schema
import json
from pathlib import Path
metrics = json.loads(Path('artifacts/metrics.json').read_text())
display(pd.DataFrame([metrics]).T.rename(columns={0:'value'}))
print(Path('artifacts/feature_schema.json').read_text())


In [ ]:
# Optional: persist the model artifact and reports in your Google Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
destination = Path('/content/drive/MyDrive/predictive-maintenance-task09')
destination.mkdir(parents=True, exist_ok=True)
for filename in ['predictive_maintenance_pipeline.joblib', 'metrics.json', 'feature_schema.json']:
    shutil.copy2(Path('artifacts') / filename, destination / filename)
print('Saved versioned Task 9 outputs to:', destination)


In [ ]:
# Smoke-test the same inference wrapper that Task 10 will reuse
from src.predict import PredictiveMaintenanceModel
model = PredictiveMaintenanceModel('artifacts/predictive_maintenance_pipeline.joblib')
sample = df[FEATURES].head(3).to_dict(orient='records')
model.predict(sample)
